# 🤖 RoboQuest 2026 — ロボットを鬼ごっこで勝たせよう！

このノートブックでは **スライダーを動かすだけ** でロボットに「鬼から逃げる」動きを学習させられます。

**手順:**
1. ▶ ボタンを押してセルを上から順番に実行してください
2. 「🤖 ロボットのパラメータ設定」のスライダーで数値を変えてみよう
3. 「🚀 学習スタート！」を実行して待つ
4. 「📊 結果を見てみよう」で動画を確認！

---
> 💡 **ヒント:** スライダーの値を変えると、ロボットの逃げ方が変わります。
> どんな値にしたら一番うまく逃げられるか、チームで考えてみよう！

In [ ]:
#@title 🔧 セットアップ（最初に一度だけ実行してください）

import subprocess, sys, os

# ── 1. GPU/CPU を自動検出してレンダリングバックエンドを設定 ──────────────────
#    mujoco を import する前に MUJOCO_GL を設定する必要がある
_has_gpu = subprocess.run('nvidia-smi', shell=True, capture_output=True).returncode == 0

if _has_gpu:
    # GPU あり → NVIDIA EGL（ハードウェアアクセラレーション）
    NVIDIA_ICD_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
    if not os.path.exists(NVIDIA_ICD_PATH):
        os.makedirs(os.path.dirname(NVIDIA_ICD_PATH), exist_ok=True)
        with open(NVIDIA_ICD_PATH, 'w') as _f:
            _f.write('{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}\n')
    os.environ['MUJOCO_GL'] = 'egl'
    print('🖥️  GPU 検出: EGL レンダリングを使用します')
else:
    # CPU のみ → OSMesa（ソフトウェアレンダリング、GPU 不要）
    subprocess.run('apt-get install -y -q libosmesa6', shell=True, check=False)
    os.environ['MUJOCO_GL'] = 'osmesa'
    print('💻  CPU モード: OSMesa レンダリングを使用します')

# ffmpeg（動画保存に必要）
subprocess.run(
    'command -v ffmpeg >/dev/null || (apt-get update -qq && apt-get install -y -q ffmpeg)',
    shell=True, check=False)

# ── 2. Python ライブラリのインストール ──────────────────────────────────────
print('ライブラリをインストール中...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'mujoco', 'gymnasium', 'stable-baselines3', 'mediapy', 'tqdm', 'pandas', 'matplotlib'],
    check=True)

# ── 3. リポジトリのクローン ─────────────────────────────────────────────────
if not os.path.exists('/content/RoboQuest2026'):
    print('リポジトリをダウンロード中...')
    subprocess.run(['git', 'clone', '-q',
        'https://github.com/tadryo/RoboQuest2026.git',
        '/content/RoboQuest2026'], check=True)

os.chdir('/content/RoboQuest2026')
if '/content/RoboQuest2026' not in sys.path:
    sys.path.insert(0, '/content/RoboQuest2026')

# ── 4. Go2 モデルのダウンロード ─────────────────────────────────────────────
print('Go2 ロボットのモデルをダウンロード中...')
subprocess.run([sys.executable, 'scripts/download_models.py'], check=True)

print('\n✅ セットアップ完了！次のセルへ進んでください。')


In [ ]:
#@title 📁 チーム名と Google Drive 接続

#@markdown チーム名を入力してください（モデルの保存先になります）
team_name = '自分のチーム名' #@param {type:"string"}

from google.colab import drive
print('Google Drive に接続します...')
drive.mount('/content/drive')

import os
SAVE_DIR = f'/content/drive/MyDrive/RoboQuest2026/{team_name}'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'\n✅ 保存先: {SAVE_DIR}')

## 🏃 フェーズ1: まず「歩く」を覚えさせよう

ロボットが鬼から逃げるためには、まず「歩く」ことを覚える必要があります。

**流れ:**
1. 歩き方の「報酬」を設定する
2. 歩行を学習させる
3. ちゃんと歩けているか確認する
4. → 次のフェーズ（鬼ごっこ）へ進む


In [ ]:
#@title 🦵 歩行パラメータの設定

#@markdown ## 速度追跡の重み
#@markdown **前後・横への速度をどれだけ正確に追う？**（大きいほど指定速度に忠実）
lin_vel_weight = 1.0 #@param {type:"slider", min:0.5, max:3.0, step:0.1}

#@markdown **回転速度をどれだけ正確に追う？**
ang_vel_weight = 0.5 #@param {type:"slider", min:0.0, max:2.0, step:0.1}

#@markdown ---
#@markdown ## 安定性の重み
#@markdown **姿勢の安定を重視する度合い**（マイナス値: 傾くほどペナルティ）
orientation_weight = -1.0 #@param {type:"slider", min:-3.0, max:0.0, step:0.1}

#@markdown **トルク（モーターの力）を節約する度合い**
torques_weight = -0.0 #@param {type:"slider", min:-0.0001, max:0.0, step:0.00001}

#@markdown **動きをなめらかにする度合い**（急な動きを減らす）
action_rate_weight = -0.05 #@param {type:"slider", min:-0.2, max:0.0, step:0.01}

#@markdown ---
#@markdown 💡 **ヒント:** まずデフォルト値で試してみよう！

print('✅ 歩行パラメータ設定完了')
print(f'  速度追跡: {lin_vel_weight}')
print(f'  回転追跡: {ang_vel_weight}')
print(f'  姿勢安定: {orientation_weight}')
print(f'  アクション滑らかさ: {action_rate_weight}')


In [ ]:
#@title ⚙️ 歩行学習の設定

#@markdown **学習ステップ数** — 大きいほど長く学習する（時間がかかる）
walk_timesteps = 300000 #@param {type:"slider", min:100000, max:1000000, step:100000}

#@markdown **並列環境数** — 同時にシミュレーションする数（多いと速い）
walk_num_envs = 4 #@param {type:"slider", min:1, max:8, step:1}

#@markdown **学習率** — 通常は変えなくてOK
walk_lr = 0.0003 #@param {type:"slider", min:0.0001, max:0.001, step:0.0001}

print(f'歩行学習設定:')
print(f'  学習ステップ数: {walk_timesteps:,}')
print(f'  並列環境数: {walk_num_envs}')
print(f'  ⏱ 目安: 約 {walk_timesteps // 100000 * 5}〜{walk_timesteps // 100000 * 15} 分')


In [ ]:
#@title 🚀 歩行学習スタート！

import os, sys
os.chdir('/content/RoboQuest2026')
if '/content/RoboQuest2026' not in sys.path:
    sys.path.insert(0, '/content/RoboQuest2026')

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import VecNormalize

from roboquest.envs.go2_walk_env import Go2WalkEnv
from roboquest.utils.reward_utils import WalkRewardConfig

walk_cfg = WalkRewardConfig(
    lin_vel_weight=lin_vel_weight,
    ang_vel_weight=ang_vel_weight,
    orientation_weight=orientation_weight,
    torques_weight=torques_weight,
    action_rate_weight=action_rate_weight,
)

def _make_walk_env():
    env = Go2WalkEnv(reward_config=walk_cfg, max_episode_steps=500)
    return Monitor(env, '/tmp/walk_monitor')

walk_vec_env = make_vec_env(_make_walk_env, n_envs=walk_num_envs)
walk_vec_env = VecNormalize(walk_vec_env, norm_obs=True, norm_reward=True)

walk_model = PPO(
    'MlpPolicy', walk_vec_env,
    learning_rate=walk_lr,
    n_steps=2048, batch_size=256, n_epochs=10,
    gamma=0.99, gae_lambda=0.95, ent_coef=0.01,
    policy_kwargs={'net_arch': [256, 256, 128]},
    verbose=0,
)

print('🏃 歩行学習を開始します...')
walk_model.learn(total_timesteps=walk_timesteps, progress_bar=True)

WALK_MODEL_PATH = '/tmp/walk_model'
walk_model.save(WALK_MODEL_PATH)
walk_vec_env.save(WALK_MODEL_PATH + '_vecnorm.pkl')
try:
    walk_model.save(f'{SAVE_DIR}/walk_model')
except Exception:
    pass
print(f'\n✅ 歩行学習完了！ → フェーズ2へ進んでください')


In [ ]:
#@title 🎬 歩行の様子を確認（斜めカメラ）

import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from roboquest.envs.go2_walk_env import Go2WalkEnv
import mujoco

walk_model_loaded = PPO.load('/tmp/walk_model')
walk_env = Go2WalkEnv(max_episode_steps=300)
walk_env.set_vel_cmd(0.8, 0.0, 0.0)  # 前進コマンド

renderer = mujoco.Renderer(walk_env.model, height=360, width=480)
frames = []
obs, _ = walk_env.reset()

for _ in range(300):
    action, _ = walk_model_loaded.predict(obs, deterministic=True)
    obs, _, terminated, truncated, _ = walk_env.step(action)
    renderer.update_scene(walk_env.data)
    frames.append(renderer.render().copy())
    if terminated or truncated:
        break

renderer.close()
walk_env.close()

import mediapy as media
media.show_video(frames, fps=30)
print(f'フレーム数: {len(frames)} ({len(frames)/30:.1f} 秒)')
print('👀 ちゃんと歩いていたら次のフェーズへ進もう！')


## 👹 フェーズ2: 鬼ごっこで1分間逃げ切れ！

歩行を学習したロボットに、今度は「逃げ方」を覚えさせます。

**ルール:**
- 5m × 5m の壁付きアリーナ
- 1分間（3000ステップ）鬼に捕まらなければ**逃げ切り！**
- 鬼は常にロボットめがけて直進してくる
- 俯瞰カメラで2体の動きが見える


In [ ]:
#@title 🏃 逃げポリシーのパラメータ設定

#@markdown ## 逃げ方の重み
#@markdown **毎ステップの生存ボーナス** — 大きいほど「生き延びること」を重視
survival_weight = 0.5 #@param {type:"slider", min:0.0, max:2.0, step:0.1}

#@markdown **鬼との距離に比例した報酬** — 大きいほど「できるだけ離れる」を重視
distance_weight = 1.0 #@param {type:"slider", min:0.0, max:3.0, step:0.1}

#@markdown **タグされた時のペナルティ** — 大きいほど「絶対つかまりたくない」に
tag_penalty = 50.0 #@param {type:"slider", min:10.0, max:100.0, step:5.0}

#@markdown ---
#@markdown ## 鬼の設定
#@markdown **鬼の速度** — 大きいほど追いかけるのが速い（難易度UP）
oni_speed = 0.025 #@param {type:"slider", min:0.01, max:0.05, step:0.005}

#@markdown ---
#@markdown 💡 **考えてみよう:** どの設定が一番長く逃げられる？

print('✅ 逃げポリシー設定完了')
print(f'  生存ボーナス: {survival_weight}')
print(f'  距離報酬: {distance_weight}')
print(f'  タグペナルティ: {tag_penalty}')
print(f'  鬼の速度: {oni_speed} m/step')


In [ ]:
#@title ⚙️ 鬼ごっこ学習の設定

#@markdown **学習ステップ数（高レベル）** — 歩行学習より少なくても大丈夫
flee_timesteps = 200000 #@param {type:"slider", min:50000, max:500000, step:50000}

#@markdown **並列環境数** — メモリが足りない場合は 1〜2 に
flee_num_envs = 2 #@param {type:"slider", min:1, max:4, step:1}

print(f'鬼ごっこ学習設定:')
print(f'  学習ステップ数: {flee_timesteps:,}')
print(f'  並列環境数: {flee_num_envs}')
print(f'  ⏱ 目安: 約 {flee_timesteps // 50000 * 3}〜{flee_timesteps // 50000 * 8} 分')


In [ ]:
#@title 👹 鬼ごっこ学習スタート！

import os, sys
os.chdir('/content/RoboQuest2026')
if '/content/RoboQuest2026' not in sys.path:
    sys.path.insert(0, '/content/RoboQuest2026')

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import VecNormalize

from roboquest.envs.go2_tag_hierarchical_env import Go2TagHierarchicalEnv
from roboquest.utils.reward_utils import FleeRewardConfig

flee_cfg = FleeRewardConfig(
    survival_weight=survival_weight,
    distance_weight=distance_weight,
    tag_penalty=tag_penalty,
)

def _make_flee_env():
    env = Go2TagHierarchicalEnv(
        low_level_model_path='/tmp/walk_model',
        flee_config=flee_cfg,
        oni_speed=oni_speed,
    )
    return Monitor(env, '/tmp/flee_monitor')

flee_vec_env = make_vec_env(_make_flee_env, n_envs=flee_num_envs)
flee_vec_env = VecNormalize(flee_vec_env, norm_obs=True, norm_reward=True)

flee_model = PPO(
    'MlpPolicy', flee_vec_env,
    learning_rate=1e-4,
    n_steps=1024, batch_size=128, n_epochs=10,
    gamma=0.99, gae_lambda=0.95, ent_coef=0.02,
    policy_kwargs={'net_arch': [128, 128]},
    verbose=0,
)

print('👹 鬼ごっこ逃げ学習を開始します...')
flee_model.learn(total_timesteps=flee_timesteps, progress_bar=True)

FLEE_MODEL_PATH = '/tmp/flee_model'
flee_model.save(FLEE_MODEL_PATH)
flee_vec_env.save(FLEE_MODEL_PATH + '_vecnorm.pkl')
try:
    flee_model.save(f'{SAVE_DIR}/flee_model')
    flee_vec_env.save(f'{SAVE_DIR}/flee_vecnorm.pkl')
except Exception:
    pass
print(f'\n✅ 鬼ごっこ学習完了！ → 結果を確認してください')


In [ ]:
#@title 📊 結果を見てみよう（俯瞰カメラ映像）

import os, sys, glob
import numpy as np
import matplotlib.pyplot as plt

os.chdir('/content/RoboQuest2026')
if '/content/RoboQuest2026' not in sys.path:
    sys.path.insert(0, '/content/RoboQuest2026')

from stable_baselines3 import PPO
from roboquest.envs.go2_tag_hierarchical_env import Go2TagHierarchicalEnv
from roboquest.utils.reward_utils import FleeRewardConfig

# ── 学習曲線 ────────────────────────────────────────────────────────────
print('📈 学習曲線を表示します...')
for log_prefix, label, color in [
    ('/tmp/walk_monitor', '歩行学習', 'steelblue'),
    ('/tmp/flee_monitor', '鬼ごっこ学習', 'tomato'),
]:
    monitor_files = glob.glob(f'{log_prefix}*.monitor.csv')
    if not monitor_files:
        continue
    import pandas as pd
    dfs = [pd.read_csv(f, skiprows=1) for f in monitor_files if True]
    dfs = [d for d in dfs if len(d) > 0]
    if not dfs:
        continue
    df_all = pd.concat(dfs).sort_values('t').reset_index(drop=True)
    rewards = df_all['r'].values
    w = min(50, len(rewards))
    smooth = np.convolve(rewards, np.ones(w)/w, mode='valid')
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(rewards, alpha=0.3, color=color)
    ax.plot(np.arange(w-1, len(rewards)), smooth, color=color, lw=2, label=label)
    ax.set_xlabel('エピソード数'); ax.set_ylabel('報酬')
    ax.set_title(f'{label} の学習曲線')
    ax.grid(alpha=0.3); ax.legend()
    plt.tight_layout(); plt.show()

# ── 俯瞰映像録画 ─────────────────────────────────────────────────────────
print('\n🎬 俯瞰カメラで鬼ごっこを録画します...')

flee_cfg = FleeRewardConfig(
    survival_weight=survival_weight,
    distance_weight=distance_weight,
    tag_penalty=tag_penalty,
)
flee_env = Go2TagHierarchicalEnv(
    low_level_model_path='/tmp/walk_model',
    flee_config=flee_cfg,
    oni_speed=oni_speed,
    render_mode='rgb_array',
)
flee_loaded = PPO.load('/tmp/flee_model')

frames = flee_env.record_episode(model=flee_loaded, fps=5)
flee_env.close()

import mediapy as media
media.show_video(frames, fps=5)

# ── 結果サマリー ─────────────────────────────────────────────────────────
print('\n📊 結果サマリー')
obs, _ = flee_env.reset() if False else (None, None)
# 簡易ロールアウトで統計収集
flee_env2 = Go2TagHierarchicalEnv(
    low_level_model_path='/tmp/walk_model',
    flee_config=flee_cfg,
    oni_speed=oni_speed,
)
obs, _ = flee_env2.reset()
distances, survived = [], 0
for _ in range(flee_env2.max_episode_steps):
    action, _ = flee_loaded.predict(obs, deterministic=True)
    obs, _, terminated, truncated, info = flee_env2.step(action)
    distances.append(info['oni_distance'])
    survived = info['survived_seconds']
    if terminated or truncated:
        break
flee_env2.close()

print(f'  生存時間: {survived:.1f} 秒 / 60.0 秒')
print(f'  鬼との平均距離: {np.mean(distances):.2f} m')
print(f'  タグされたか: {"はい" if info.get("is_tagged") else "いいえ（逃げきり！🎉）"}')
if survived >= 60:
    print('\n🎉🎉 1分間逃げ切り成功！！ 🎉🎉')


In [ ]:
#@title 💾 モデルを保存して大会に提出

import os, json

params = {
    'team_name': team_name,
    'walk': {
        'lin_vel_weight': lin_vel_weight,
        'ang_vel_weight': ang_vel_weight,
        'orientation_weight': orientation_weight,
        'action_rate_weight': action_rate_weight,
        'timesteps': walk_timesteps,
    },
    'flee': {
        'survival_weight': survival_weight,
        'distance_weight': distance_weight,
        'tag_penalty': tag_penalty,
        'oni_speed': oni_speed,
        'timesteps': flee_timesteps,
    },
}

try:
    with open(f'{SAVE_DIR}/params.json', 'w', encoding='utf-8') as f:
        json.dump(params, f, ensure_ascii=False, indent=2)
    print(f'✅ パラメータを保存: {SAVE_DIR}/params.json')
    print(f'✅ 歩行モデル: {SAVE_DIR}/walk_model.zip')
    print(f'✅ 逃げモデル: {SAVE_DIR}/flee_model.zip')
    print(f'\n🎉 大会への提出準備完了！')
except Exception as e:
    print(f'Drive 保存エラー: {e}')
    print('ローカル保存済み: /tmp/walk_model.zip, /tmp/flee_model.zip')
